# Analyse per signal med zoom

Denne notatboken er laget for at sensor enkelt skal kunne kjøre analysen.

Den gir bare ut:

- en kort tabell med bølgeparametere
- ett felles plott for de fem bølgetilfellene fra de valgte tidsvinduene

## Mappestruktur

```text
Analyse_pr_signal_med_zoom_sensor_ready.ipynb
analyze_wave_signals.py
sensor_data_wave_zoom/
    wave_data/
        w60.txt
        w66.txt
        w70.txt
        w76.txt
        w80.txt
    results/
```


In [1]:
from pathlib import Path
import csv
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from analyze_wave_signals import AnalysisWindow, collect_signal_records, extract_analysis_window, summarize_record

DATA_DIR = NOTEBOOK_DIR / "sensor_data_wave_zoom" / "wave_data"
RESULT_DIR = NOTEBOOK_DIR / "sensor_data_wave_zoom" / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_WINDOWS = {
    "W60": AnalysisWindow(6.0, 10.0),
    "W66": AnalysisWindow(6.0, 10.0),
    "W70": AnalysisWindow(6.0, 10.0),
    "W76": AnalysisWindow(6.0, 10.0),
    "W80": AnalysisWindow(6.0, 10.0),
}

records = sorted(
    collect_signal_records(DATA_DIR),
    key=lambda record: record.wave_strength,
)

print(f"Notebook folder: {NOTEBOOK_DIR}")
print(f"Data folder:     {DATA_DIR}")
print(f"Results folder:  {RESULT_DIR}")
print("Loaded signals:")
for record in records:
    window = ANALYSIS_WINDOWS.get(record.label)
    print(f" - {record.label}: time = {record.times[0]:.3f} to {record.times[-1]:.3f} s | window = {window.start_s:.3f} to {window.end_s:.3f} s")


Notebook folder: C:\Users\knutm\Documents\Skole\Master\Python\bølge
Data folder:     C:\Users\knutm\Documents\Skole\Master\Python\bølge\sensor_data_wave_zoom\wave_data
Results folder:  C:\Users\knutm\Documents\Skole\Master\Python\bølge\sensor_data_wave_zoom\results
Loaded signals:
 - W60: time = 0.000 to 12.830 s | window = 6.000 to 10.000 s
 - W66: time = 0.000 to 13.080 s | window = 6.000 to 10.000 s
 - W70: time = 0.000 to 10.820 s | window = 6.000 to 10.000 s
 - W76: time = 0.000 to 12.980 s | window = 6.000 to 10.000 s
 - W80: time = 0.000 to 12.720 s | window = 6.000 to 10.000 s


In [2]:
summary_rows = []

def local_smooth_signal(values, window_size=9):
    values = np.asarray(values, dtype=float)
    if window_size <= 1:
        return values.copy()
    kernel = np.ones(window_size, dtype=float) / window_size
    return np.convolve(values, kernel, mode="same")

def interpolate_zero_crossing_time(times, values, left_index, right_index):
    t0 = float(times[left_index])
    t1 = float(times[right_index])
    y0 = float(values[left_index])
    y1 = float(values[right_index])
    if y0 == 0.0:
        return t0
    if y1 == 0.0:
        return t1
    if y1 == y0:
        return 0.5 * (t0 + t1)
    fraction = -y0 / (y1 - y0)
    return float(t0 + fraction * (t1 - t0))

def find_zero_crossings(times, values):
    upward = []
    downward = []
    for index in range(len(values) - 1):
        y0 = float(values[index])
        y1 = float(values[index + 1])
        if y0 <= 0.0 < y1 or y0 < 0.0 <= y1:
            upward.append((interpolate_zero_crossing_time(times, values, index, index + 1), index, index + 1))
        elif y0 >= 0.0 > y1 or y0 > 0.0 >= y1:
            downward.append((interpolate_zero_crossing_time(times, values, index, index + 1), index, index + 1))
    return upward, downward

for record in records:
    window = ANALYSIS_WINDOWS.get(record.label)
    row = summarize_record(record, analysis_window=window)

    times_window, values_window_cm = extract_analysis_window(record, analysis_window=window)
    centered_window_cm = np.asarray(values_window_cm, dtype=float) - float(np.mean(values_window_cm))
    relative_times_s = times_window - float(times_window[0])
    smooth_centered_cm = local_smooth_signal(centered_window_cm, window_size=9)
    upward_crossings, downward_crossings = find_zero_crossings(relative_times_s, smooth_centered_cm)

    peak_back_widths_s = []
    peak_front_widths_s = []

    for wave_index in range(len(upward_crossings) - 1):
        back_time_s, back_left_index, back_right_index = upward_crossings[wave_index]
        next_back_time_s, _, _ = upward_crossings[wave_index + 1]
        front_candidates = [item for item in downward_crossings if back_time_s < item[0] < next_back_time_s]
        if not front_candidates:
            continue
        front_time_s, front_left_index, _ = front_candidates[0]
        crest_slice_start = back_right_index
        crest_slice_end = front_left_index + 1
        if crest_slice_end <= crest_slice_start:
            continue
        crest_local_index = int(np.argmax(centered_window_cm[crest_slice_start:crest_slice_end]))
        crest_index = crest_slice_start + crest_local_index
        crest_time_s = float(relative_times_s[crest_index])
        crest_value_cm = float(centered_window_cm[crest_index])
        if not (back_time_s < crest_time_s < front_time_s < next_back_time_s):
            continue
        if crest_value_cm <= 0:
            continue
        wb_s = float(crest_time_s - back_time_s)
        wf_s = float(front_time_s - crest_time_s)
        if wb_s <= 0 or wf_s <= 0:
            continue
        peak_back_widths_s.append(wb_s)
        peak_front_widths_s.append(wf_s)

    R_wb_wf = (
        float(np.mean(peak_back_widths_s) / np.mean(peak_front_widths_s))
        if peak_back_widths_s and peak_front_widths_s and np.mean(peak_front_widths_s) > 0
        else math.nan
    )

    summary_rows.append({
        "Wave case": record.label,
        "a [cm]": row["amplitude_cm"],
        "T [s]": row["period_s"],
        "f [Hz]": row["frequency_hz"],
        "k [1/m]": row["wave_number_m_inv"],
        "lambda [m]": row["wavelength_m"],
        "R_wb/wf [-]": R_wb_wf,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Wave case").reset_index(drop=True)
display(summary_df.round(6))

summary_df.to_csv(RESULT_DIR / "wave_summary_table.csv", index=False, encoding="utf-8-sig")
print(f"Saved table: {RESULT_DIR / 'wave_summary_table.csv'}")


,Wave case,a [cm],T [s],f [Hz],k [1/m],lambda [m],R_wb/wf [-]
0,W60,0.633978,0.503510,1.986058,15.875880,0.395769,0.901770
1,W66,0.715815,0.456270,2.191686,19.331052,0.325031,1.355898
2,W70,0.789365,0.443432,2.255137,20.466365,0.307001,1.062733
3,W76,0.801796,0.404106,2.474597,24.643375,0.254964,1.156295
4,W80,0.903315,0.396204,2.523950,25.636119,0.245091,1.446243


Saved table: C:\Users\knutm\Documents\Skole\Master\Python\bølge\sensor_data_wave_zoom\results\wave_summary_table.csv


In [3]:
fig, axes = plt.subplots(len(records), 1, figsize=(14, 3.6 * len(records)), sharex=True)
if len(records) == 1:
    axes = [axes]

for ax, record in zip(axes, records):
    window = ANALYSIS_WINDOWS.get(record.label)
    times_window, values_window_cm = extract_analysis_window(record, analysis_window=window, center_on_mean=False)
    relative_times_s = times_window - float(times_window[0])
    ax.plot(relative_times_s, values_window_cm, linewidth=1.6, label=record.label)
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1.0, alpha=0.6)
    ax.set_title(record.label, fontsize=12, pad=6)
    ax.set_ylabel("Distance [cm]")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Time from window start [s]")
fig.suptitle("Common plot for all five wave cases from the selected windows", fontsize=16, fontweight="bold", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.985))

plot_path = RESULT_DIR / "common_wave_plot.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved plot: {plot_path}")


Saved plot: C:\Users\knutm\Documents\Skole\Master\Python\bølge\sensor_data_wave_zoom\results\common_wave_plot.png
